<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 5.6: A Five-Stage Pipeline
**Prev: [The Official RISC-V Tests](5.5_compliance.ipynb)**<br>
**Next: [Hazards](5.7_hazards.ipynb)**<br>

## Motivation
The single-cycle processor of [Module 5.4](5.4_single_cycle.ipynb) does all the work of an
instruction in one clock cycle, so the clock period must cover the slowest instruction from start
to end: fetch, decode, register read, ALU, memory, write back. A **pipeline** cuts that path into
stages with registers between them, and works on several instructions at once, one per stage, like
an assembly line: the clock period only has to cover the slowest stage, and an instruction still
finishes every cycle.

This notebook builds the classic five-stage pipeline: fetch (IF), decode and register read (ID),
execute (EX), memory (MEM) and write back (WB). It also meets the price of overlapping
instructions: *hazards*. An instruction may need a result that an older one has not written yet
(a data hazard), and the instructions fetched after a jump may not be the ones that run next (a
control hazard). This notebook handles control hazards and halts; data hazards it avoids with
nops, and the next one handles them in hardware.

## Setup

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
interp.load.module(os.pwd / "source" / "load-riscv.sc")

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.simulator.PeekPokeAPI._
import riscv._

---
# Stages and stage registers

Each stage passes its instruction to the next through a register: IF/ID, ID/EX, EX/MEM and MEM/WB,
each a `Bundle` of what the later stages need, wrapped in `Valid` (a stage may hold no instruction:
a *bubble*). `Stage` builds such a register: it takes the next stage's input at each clock edge,
or a bubble when it is *flushed*, and has an enable for the stalls of the next notebook (Chisel's
`RegEnable` does the same without the flush). The datapath blocks are the ones of 5.2-5.4.

The interesting part is what happens between the stages:

- **EX** resolves jumps and branches. The pipeline has fetched two more instructions by then (at
  pc + 4 and pc + 8, a guess that the branch is not taken); if the branch is taken, they are on
  the wrong path and must be *flushed*, and fetch restarts at the target.
- **MEM** is the last stage that can find a reason to halt (a misaligned or out-of-range access).
  An instruction that halts the core must not let any younger instruction change memory (in MEM)
  or registers (in WB), so it flushes them all and fetching stops.
- **WB** writes the register file and reports the instruction on the retire port, which the
  golden model compares as for the single-cycle core.

In [ ]:
println(os.read(os.pwd / "riscv" / "src" / "main" / "scala" / "riscv" / "PipelinedCore.scala"))

---
# Exercise: the flushes

`riscv.PipelineFlush` decides which stage registers take a bubble: `redirect` is a jump or taken
branch in EX, `halt` a halting instruction in MEM. Complete
`riscv/src/main/scala/riscv/PipelineFlush.scala`, restart the kernel and run the notebook again
(until then, the cells that build the core stop with a `NotImplementedError`, as do those of 5.2-5.4
if their exercises are not done). Think about which instructions are younger than the one in EX,
and than the one in MEM, and where each of them is.

In [ ]:
simulate(new PipelineFlush) { c =>
  for (redirect <- Seq(false, true); halt <- Seq(false, true)) {
    c.io.redirect.poke(redirect.B)
    c.io.halt.poke(halt.B)
    c.io.flushIfId.expect((redirect || halt).B, s"redirect $redirect, halt $halt")
    c.io.flushIdEx.expect((redirect || halt).B, s"redirect $redirect, halt $halt")
    c.io.flushExMem.expect(halt.B, s"redirect $redirect, halt $halt: the jump itself goes on")
  }
}
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  io.flushIfId  := io.redirect || io.halt
  io.flushIdEx  := io.redirect || io.halt
  io.flushExMem := io.halt // the jump or branch in EX itself goes on
</pre></article></div></section></div>

---
# Data hazards, and nops

An instruction reads its registers in ID and its result is written in WB, three stages later. If
the next instruction uses that result, it reads the register too early and gets the old value:

In [ ]:
val program = """
  li    a0, 5
  addi  a1, a0, 1     # needs a0 = 5
  ecall
"""

In [ ]:
simulate(new PipelinedSystem) { c =>
  println(CoSim.compare(c, Asm(program)).getOrElse("no difference"))
}

`addi` read `a0` while `li` was still in EX. The register file is written in the first half of a
cycle and read in the second, so a result can be read in the cycle its instruction is in WB: an
instruction that uses it must come three instructions later, with two others between. Until the
hardware handles it, the program can make room with `nop`s. `Asm.padded(source, 2)` assembles a
program with two nops after every instruction (inside pseudo-instructions too), which is enough for
any program:

In [ ]:
simulate(new PipelinedSystem) { c =>
  println(CoSim.compare(c, Asm.padded(program, 2)).getOrElse("no difference"))
}

---
# Running programs

The Fibonacci and factorial programs, padded, retire exactly as on the golden model, and so do
random programs. The cycle count shows the cost: the nops are instructions too, each taken branch
or jump wastes two cycles on flushed instructions, and filling the pipeline takes four.

In [ ]:
def cycles(c: PipelinedSystem, image: Seq[Long]): (Int, Int) = {
  CoSim.load(c, image)
  var n = 0
  while (!c.io.halted.peek().litToBoolean) { c.clock.step(); n += 1 }
  (n, new IsaSim(image).run().size) // cycles, instructions
}
simulate(new PipelinedSystem) { c =>
  for (name <- Seq("fib", "fact")) {
    val source = Programs.source("programs/src", name)
    CoSim.compare(c, Asm.padded(source, 2)).foreach(d => sys.error(s"$name: $d"))
    val useful       = new IsaSim(Asm(source)).run().size
    val (n, retired) = cycles(c, Asm.padded(source, 2))
    println(f"$name%-5s $useful%4d instructions, $retired%4d with the nops, $n%4d cycles")
  }
  for (seed <- 1L to 5L)
    CoSim.compare(c, Asm.padded(RandomProgram(seed), 2)).foreach(d => sys.error(s"RandomProgram($seed): $d"))
}
println("SUCCESS!!")

---
# Halting

A halting instruction flushes everything behind it, so nothing younger takes effect. In this
program (not padded: the two nops give `auipc`'s result time to reach the `sw`), the first store
is in EX and the second in ID when `ecall` reaches MEM. Both are flushed before they reach MEM,
and nothing retires after `ecall`: `CoSim.compare` checks that too, a few cycles past the halt.

In [ ]:
simulate(new PipelinedSystem) { c =>
  val source = "auipc t0, 0\nnop\nnop\necall\nsw t0, 0(t0)\nsw t0, 4(t0)"
  println(CoSim.compare(c, Asm(source)).getOrElse("no difference: the stores never happened"))
}

---
# And the official tests?

The riscv-tests of [Module 5.5](5.5_compliance.ipynb) are full of back-to-back dependent
instructions, so they fail on this pipeline as they are; padded with nops they grow so much that
their branches to `fail` no longer reach (a branch reaches 4 KB). The [next notebook](5.7_hazards.ipynb) removes the
need for nops: it forwards results from the later stages to EX and stalls when it must, and then
the official tests run unchanged.

---
# You're done!

[Return to the top.](#top)